### Data Preperation for LMVA dataset

`This jupyter notebook is focused on filtering, chunking, & cleaning the data such that we can directly use it for steamlit dashboard...`

In [25]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

`load the data`

In [2]:
dflm=pd.read_csv("sail_lvma.csv")
dfsl=pd.read_csv("sensor_location_clean.csv")

`prepare the timestamp`
> change from str to datetime

In [4]:
dflm['timestamp']=pd.to_datetime(dflm['timestamp'],errors="coerce")

`converting sensor counts into dashboard-ready format`

In [6]:
# Keep only actual sensor-direction columns
metadata_cols = {
    "timestamp", "hour", "minute", "day",
    "month", "weekday", "is_weekend"
}
sensor_cols = [c for c in dflm.columns if c not in metadata_cols]

# Reshape wide data into one row per timestamp and sensor-direction
ped_counts = dflm.melt(
    id_vars=["timestamp"],
    value_vars=sensor_cols,
    var_name="sensor_direction",
    value_name="pedestrian_count"
)

# Split only at the final underscore:
# GASA-06-B_95 -> sensor_id=GASA-06-B, direction=95
ped_counts[["sensor_id", "direction"]] = (
    ped_counts["sensor_direction"].str.rsplit("_", n=1, expand=True)
)

ped_counts["pedestrian_count"] = pd.to_numeric(
    ped_counts["pedestrian_count"], errors="coerce"
)

ped_counts = ped_counts.dropna(
    subset=["timestamp", "pedestrian_count"]
)

`basically pivot the data, tranpose it for dashboard`

In [11]:
ped_counts.iloc[2390:2410]

,timestamp,sensor_direction,pedestrian_count,sensor_id,direction
2390,2025-08-24 23:30:00+02:00,CMSA-GAKH-01_0,12,CMSA-GAKH-01,0
2391,2025-08-24 23:33:00+02:00,CMSA-GAKH-01_0,5,CMSA-GAKH-01,0
2392,2025-08-24 23:36:00+02:00,CMSA-GAKH-01_0,0,CMSA-GAKH-01,0
2393,2025-08-24 23:39:00+02:00,CMSA-GAKH-01_0,10,CMSA-GAKH-01,0
2394,2025-08-24 23:42:00+02:00,CMSA-GAKH-01_0,5,CMSA-GAKH-01,0
2395,2025-08-24 23:45:00+02:00,CMSA-GAKH-01_0,11,CMSA-GAKH-01,0
2396,2025-08-24 23:48:00+02:00,CMSA-GAKH-01_0,2,CMSA-GAKH-01,0
2397,2025-08-24 23:51:00+02:00,CMSA-GAKH-01_0,10,CMSA-GAKH-01,0
2398,2025-08-24 23:54:00+02:00,CMSA-GAKH-01_0,3,CMSA-GAKH-01,0
2399,2025-08-24 23:57:00+02:00,CMSA-GAKH-01_0,6,CMSA-GAKH-01,0


`now its easier to attach location to each sensor`

In [12]:
ped_counts = ped_counts.merge(
    dfsl,
    left_on="sensor_id",
    right_on="object_number",
    how="left",
    validate="many_to_one"
)

print("Prepared rows:", len(ped_counts))
print("Unmatched sensor rows:", ped_counts["location_name"].isna().sum())

display(ped_counts.head())

Prepared rows: 177600
Unmatched sensor rows: 4800


,timestamp,sensor_direction,pedestrian_count,sensor_id,direction,object_number,location_name,Latitude,Longitude,width,effective_width
0,2025-08-20 00:00:00+02:00,CMSA-GAKH-01_0,15,CMSA-GAKH-01,0,CMSA-GAKH-01,Kalverstraat t.h.v. 1,52.372634,4.892071,8.0,6.7
1,2025-08-20 00:03:00+02:00,CMSA-GAKH-01_0,1,CMSA-GAKH-01,0,CMSA-GAKH-01,Kalverstraat t.h.v. 1,52.372634,4.892071,8.0,6.7
2,2025-08-20 00:06:00+02:00,CMSA-GAKH-01_0,5,CMSA-GAKH-01,0,CMSA-GAKH-01,Kalverstraat t.h.v. 1,52.372634,4.892071,8.0,6.7
3,2025-08-20 00:09:00+02:00,CMSA-GAKH-01_0,4,CMSA-GAKH-01,0,CMSA-GAKH-01,Kalverstraat t.h.v. 1,52.372634,4.892071,8.0,6.7
4,2025-08-20 00:12:00+02:00,CMSA-GAKH-01_0,4,CMSA-GAKH-01,0,CMSA-GAKH-01,Kalverstraat t.h.v. 1,52.372634,4.892071,8.0,6.7


In [18]:
ped_counts.loc[7197:7203]

,timestamp,sensor_direction,pedestrian_count,sensor_id,direction,object_number,location_name,Latitude,Longitude,width,effective_width
7197,2025-08-24 23:51:00+02:00,CMSA-GAWW-11_120,51,CMSA-GAWW-11,120,CMSA-GAWW-11,Korte Niezel,52.374616,4.89983,3.8,3.4
7198,2025-08-24 23:54:00+02:00,CMSA-GAWW-11_120,24,CMSA-GAWW-11,120,CMSA-GAWW-11,Korte Niezel,52.374616,4.89983,3.8,3.4
7199,2025-08-24 23:57:00+02:00,CMSA-GAWW-11_120,38,CMSA-GAWW-11,120,CMSA-GAWW-11,Korte Niezel,52.374616,4.89983,3.8,3.4
7200,2025-08-20 00:00:00+02:00,CMSA-GAWW-11_300,33,CMSA-GAWW-11,300,CMSA-GAWW-11,Korte Niezel,52.374616,4.89983,3.8,3.4
7201,2025-08-20 00:03:00+02:00,CMSA-GAWW-11_300,29,CMSA-GAWW-11,300,CMSA-GAWW-11,Korte Niezel,52.374616,4.89983,3.8,3.4
7202,2025-08-20 00:06:00+02:00,CMSA-GAWW-11_300,22,CMSA-GAWW-11,300,CMSA-GAWW-11,Korte Niezel,52.374616,4.89983,3.8,3.4
7203,2025-08-20 00:09:00+02:00,CMSA-GAWW-11_300,47,CMSA-GAWW-11,300,CMSA-GAWW-11,Korte Niezel,52.374616,4.89983,3.8,3.4


In [20]:
display(
    ped_counts[ped_counts["sensor_id"] == "GASA-06-B"]
    .groupby("direction")["pedestrian_count"]
    .agg(["count", "sum", "max"])
)
# correct. . . matches with the EDA.

,count,sum,max
direction,,,
275,2400,7792,83
95,2400,11270,164


`export the data. . .`
> using the paraquet to export the final file, because it is efficient for reading and good for streamlit data

In [26]:
output_dir = Path("../Streamlit and Dataset/Final Data")
output_dir.mkdir(parents=True, exist_ok=True)  # if doesn't exist make it

ped_counts.to_parquet(
    output_dir / "pedestrian_counts.parquet",
    index=False
)

print("Prepared files saved successfully.")

Prepared files saved successfully.
